# Shell Hackathon EDA

Exploratory analysis for blend property prediction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")
train_df.head()

## Data Overview

In [ ]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

display(train_df.dtypes.to_frame("dtype").T)

missing = train_df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing"))

duplicates = train_df.duplicated().sum()
print("Duplicate rows:", duplicates)

## Target Distribution

In [ ]:
targets = [f"BlendProperty{i}" for i in range(1, 11)]

for target in targets:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sns.histplot(train_df[target], bins=30, kde=True, ax=axes[0])
    axes[0].set_title(f"{target} Histogram")
    sns.boxplot(x=train_df[target], ax=axes[1])
    axes[1].set_title(f"{target} Box Plot")
    plt.tight_layout()
    plt.show()

## Component Fractions

In [ ]:
fraction_cols = [f"Component{i}_fraction" for i in range(1, 6)]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for idx, col in enumerate(fraction_cols):
    sns.histplot(train_df[col], bins=25, ax=axes[idx])
    axes[idx].set_title(col)
plt.tight_layout()
plt.show()

fraction_sum = train_df[fraction_cols].sum(axis=1)
print("Fraction sum mean:", fraction_sum.mean())
print("Fraction sum min/max:", fraction_sum.min(), fraction_sum.max())
sns.histplot(fraction_sum, bins=30)
plt.title("Fraction Sum Distribution")
plt.show()

## Property Correlations

In [ ]:
property_cols = [
    f"Component{i}_Property{j}" for i in range(1, 6) for j in range(1, 11)
]
corr = train_df[property_cols + targets].corr()
heatmap_data = corr.loc[property_cols, targets]
plt.figure(figsize=(8, 12))
sns.heatmap(heatmap_data, cmap="coolwarm", center=0)
plt.title("Property vs Target Correlations")
plt.tight_layout()
plt.show()

## Mixing Rule Validation

In [ ]:
linmix = pd.DataFrame(index=train_df.index)
for prop_idx in range(1, 11):
    mix = 0.0
    for comp_idx in range(1, 6):
        mix += (
            train_df[f"Component{comp_idx}_fraction"]
            * train_df[f"Component{comp_idx}_Property{prop_idx}"]
        )
    linmix[f"LinMix_P{prop_idx}"] = mix

for prop_idx in range(1, 11):
    target = f"BlendProperty{prop_idx}"
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sns.scatterplot(x=linmix[f"LinMix_P{prop_idx}"], y=train_df[target], ax=axes[0], s=12)
    axes[0].set_xlabel(f"LinMix_P{prop_idx}")
    axes[0].set_ylabel(target)
    axes[0].set_title("Linear Mix vs Actual")

    residuals = train_df[target] - linmix[f"LinMix_P{prop_idx}"]
    sns.histplot(residuals, bins=30, ax=axes[1])
    axes[1].set_title("Residuals")
    plt.tight_layout()
    plt.show()

## Observations

Linear mixing provides a strong baseline but misses nonlinear effects for several targets.

The component fraction distributions are well-behaved and typically sum to one, suggesting limited data quality issues.

Correlations between component properties and targets vary by property index, supporting per-target modeling.